# Three-Model Ablation Comparison V2

This notebook compares the three completed V2 experiments on the **same dataset
intersection**:

1. Full fractional Two-Tower;
2. Correlation Two-Tower ablation;
3. Transformer-only ablation.

It performs the following steps without modifying the original result files:

- validates columns, completed rows, duplicate datasets, sample counts, and the
  expected 124-dataset common universe;
- reproduces simple and sample-weighted aggregate metrics;
- computes paired per-dataset improvements, win/tie/loss counts, bootstrap 95%
  confidence intervals, Wilcoxon signed-rank tests, Holm-adjusted p-values, and
  rank-biserial effect sizes;
- computes Friedman tests and average model ranks;
- exports comparison CSV/JSON files and publication-ready diagnostic figures.

## Statistical convention

Every paired `improvement` value is defined so that **positive always favors
model A**:

- AUC/ACC: `model_A - model_B`;
- Loss: `model_B - model_A` because lower loss is better.

The notebook reports both two-sided Wilcoxon p-values and the directional
one-sided hypothesis that model A is better. Holm correction is applied across
all reported pairwise tests of the same p-value family.

In [ ]:
# Cell 1 - imports and paths
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


# -----------------------------------------------------------------------------
# EDIT ONLY RUN_ROOT IF YOUR final_runs_v2 DIRECTORY IS ELSEWHERE
# -----------------------------------------------------------------------------
RUN_ROOT = Path(
    r"D:\2025暑期科研\UCRArchive_2018\TwoTower_Project_V2\final_runs_v2"
)

RESULT_FILES = {
    "twotower": RUN_ROOT / "twotower" / "seed_42" / "final_results.csv",
    "correlation": RUN_ROOT
    / "correlation_aligned_v2"
    / "seed_42"
    / "final_results.csv",
    "transformer_only": RUN_ROOT
    / "transformer_only"
    / "seed_42"
    / "final_results.csv",
}

MODEL_ORDER = ["twotower", "correlation", "transformer_only"]
MODEL_LABEL = {
    "twotower": "Fractional Two-Tower",
    "correlation": "Correlation Two-Tower",
    "transformer_only": "Transformer-only",
}

OUTPUT_DIR = RUN_ROOT / "ablation_comparison_v2" / "seed_42"
FIGURE_DIR = OUTPUT_DIR / "figures"
EXPECTED_COMMON_DATASETS = 124
STRICT_COMMON_DATASETS = True
TIE_TOLERANCE = 1e-12
BOOTSTRAP_REPEATS = 10_000
BOOTSTRAP_SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Run root:", RUN_ROOT)
for model_name, path in RESULT_FILES.items():
    print(f"{MODEL_LABEL[model_name]}: {path}")
print("Comparison output:", OUTPUT_DIR)

In [ ]:
# Cell 2 - reusable validation and statistical helpers
REQUIRED_COLUMNS = {"dataset", "test_auc", "test_acc", "test_loss", "n_samples"}
METRIC_SPECS = {
    "test_auc": {"label": "AUC", "higher_is_better": True},
    "test_acc": {"label": "ACC", "higher_is_better": True},
    "test_loss": {"label": "Loss", "higher_is_better": False},
}
PAIR_ORDER = [
    ("twotower", "correlation"),
    ("twotower", "transformer_only"),
    ("correlation", "transformer_only"),
]


def load_and_validate_result(model_name: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {MODEL_LABEL[model_name]} CSV: {path}")

    df = pd.read_csv(path)
    missing = sorted(REQUIRED_COLUMNS.difference(df.columns))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")

    if "status" in df.columns:
        unexpected = sorted(set(df["status"].dropna().astype(str)) - {"completed"})
        if unexpected:
            raise ValueError(
                f"{MODEL_LABEL[model_name]} final_results contains non-completed "
                f"statuses: {unexpected}"
            )
        df = df[df["status"] == "completed"].copy()
    else:
        df = df.copy()

    if df["dataset"].isna().any():
        raise ValueError(f"{MODEL_LABEL[model_name]} contains a missing dataset name.")
    df["dataset"] = df["dataset"].astype(str)

    duplicated = df.loc[df["dataset"].duplicated(keep=False), "dataset"].tolist()
    if duplicated:
        raise ValueError(
            f"{MODEL_LABEL[model_name]} has duplicate dataset rows: {sorted(set(duplicated))}"
        )

    for column in ["test_auc", "test_acc", "test_loss", "n_samples"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    if df[["test_acc", "test_loss", "n_samples"]].isna().any().any():
        raise ValueError(
            f"{MODEL_LABEL[model_name]} has missing/non-numeric ACC, loss, or sample count."
        )
    if (df["n_samples"] <= 0).any():
        raise ValueError(f"{MODEL_LABEL[model_name]} contains a non-positive n_samples.")

    return df.sort_values("dataset").reset_index(drop=True)


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    valid = values.notna() & weights.notna()
    if not valid.any():
        return float("nan")
    return float(np.average(values[valid].to_numpy(), weights=weights[valid].to_numpy()))


def bootstrap_mean_ci(
    values: np.ndarray,
    repeats: int = BOOTSTRAP_REPEATS,
    seed: int = BOOTSTRAP_SEED,
) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, values.size, size=(repeats, values.size))
    means = values[indices].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return float(low), float(high)


def holm_adjust(p_values: pd.Series) -> pd.Series:
    values = p_values.to_numpy(dtype=float)
    adjusted = np.full(values.shape, np.nan, dtype=float)
    valid_positions = np.flatnonzero(np.isfinite(values))
    if valid_positions.size == 0:
        return pd.Series(adjusted, index=p_values.index)

    valid_values = values[valid_positions]
    order = np.argsort(valid_values)
    sorted_positions = valid_positions[order]
    sorted_p = valid_values[order]
    m = len(sorted_p)
    sorted_adjusted = np.maximum.accumulate(
        np.asarray([(m - rank) * p for rank, p in enumerate(sorted_p)])
    )
    sorted_adjusted = np.minimum(sorted_adjusted, 1.0)
    adjusted[sorted_positions] = sorted_adjusted
    return pd.Series(adjusted, index=p_values.index)


def rank_biserial_from_differences(improvement: np.ndarray) -> float:
    improvement = np.asarray(improvement, dtype=float)
    nonzero = improvement[np.abs(improvement) > TIE_TOLERANCE]
    if nonzero.size == 0:
        return 0.0
    ranks = stats.rankdata(np.abs(nonzero), method="average")
    positive = float(ranks[nonzero > 0].sum())
    negative = float(ranks[nonzero < 0].sum())
    return (positive - negative) / (positive + negative)


def safe_wilcoxon(improvement: np.ndarray, alternative: str) -> tuple[float, float]:
    improvement = np.asarray(improvement, dtype=float)
    if improvement.size == 0 or np.all(np.abs(improvement) <= TIE_TOLERANCE):
        return 0.0, 1.0
    result = stats.wilcoxon(
        improvement,
        zero_method="wilcox",
        correction=False,
        alternative=alternative,
        method="auto",
    )
    return float(result.statistic), float(result.pvalue)

In [ ]:
# Cell 3 - load files and enforce a fair common-dataset comparison
results = {
    model_name: load_and_validate_result(model_name, RESULT_FILES[model_name])
    for model_name in MODEL_ORDER
}

dataset_sets = {
    model_name: set(df["dataset"])
    for model_name, df in results.items()
}
dataset_union = sorted(set.union(*dataset_sets.values()))
common_datasets = sorted(set.intersection(*dataset_sets.values()))

presence = pd.DataFrame({"dataset": dataset_union})
for model_name in MODEL_ORDER:
    presence[model_name] = presence["dataset"].isin(dataset_sets[model_name])
presence.to_csv(OUTPUT_DIR / "dataset_presence.csv", index=False)

print("\nCompleted rows by model:")
for model_name in MODEL_ORDER:
    print(f"  {MODEL_LABEL[model_name]}: {len(results[model_name])}")
print("Common datasets:", len(common_datasets))

sets_identical = all(dataset_sets[name] == dataset_sets[MODEL_ORDER[0]] for name in MODEL_ORDER[1:])
if not sets_identical:
    for model_name in MODEL_ORDER:
        missing_from_model = sorted(set(dataset_union) - dataset_sets[model_name])
        print(f"Missing from {MODEL_LABEL[model_name]}: {missing_from_model}")
    if STRICT_COMMON_DATASETS:
        raise ValueError(
            "The three final_results.csv files do not contain identical dataset names. "
            "Inspect dataset_presence.csv before comparing."
        )

if len(common_datasets) != EXPECTED_COMMON_DATASETS:
    raise ValueError(
        f"Expected {EXPECTED_COMMON_DATASETS} common datasets, found {len(common_datasets)}."
    )

indexed = {
    model_name: df.set_index("dataset").loc[common_datasets].copy()
    for model_name, df in results.items()
}

sample_counts = pd.concat(
    [indexed[name]["n_samples"].rename(name) for name in MODEL_ORDER],
    axis=1,
)
if not sample_counts.eq(sample_counts.iloc[:, 0], axis=0).all().all():
    bad = sample_counts[~sample_counts.eq(sample_counts.iloc[:, 0], axis=0).all(axis=1)]
    raise ValueError(f"n_samples differs across models:\n{bad}")

print("Dataset names and n_samples are identical across all three models.")
print("AUC missing counts:")
for model_name in MODEL_ORDER:
    print(f"  {MODEL_LABEL[model_name]}: {indexed[model_name]['test_auc'].isna().sum()}")

In [ ]:
# Cell 4 - reproduce aggregate metrics
aggregate_rows = []
for model_name in MODEL_ORDER:
    df = indexed[model_name]
    aggregate_rows.append(
        {
            "model": model_name,
            "model_label": MODEL_LABEL[model_name],
            "datasets_completed": int(len(df)),
            "auc_valid_datasets": int(df["test_auc"].notna().sum()),
            "simple_mean_auc": float(df["test_auc"].mean()),
            "simple_mean_acc": float(df["test_acc"].mean()),
            "simple_mean_loss": float(df["test_loss"].mean()),
            "weighted_auc": weighted_mean(df["test_auc"], df["n_samples"]),
            "weighted_acc": weighted_mean(df["test_acc"], df["n_samples"]),
            "weighted_loss": weighted_mean(df["test_loss"], df["n_samples"]),
        }
    )

aggregate_summary = pd.DataFrame(aggregate_rows)
aggregate_summary.to_csv(OUTPUT_DIR / "aggregate_summary.csv", index=False)

display_columns = [
    "model_label",
    "datasets_completed",
    "simple_mean_auc",
    "simple_mean_acc",
    "simple_mean_loss",
    "weighted_auc",
    "weighted_acc",
]
print(aggregate_summary[display_columns].round(6).to_string(index=False))

In [ ]:
# Cell 5 - paired improvements, W/T/L, bootstrap CI, and Wilcoxon tests
pairwise_rows = []

for pair_index, (model_a, model_b) in enumerate(PAIR_ORDER):
    for metric_name, spec in METRIC_SPECS.items():
        paired = pd.concat(
            [
                indexed[model_a][metric_name].rename("a"),
                indexed[model_b][metric_name].rename("b"),
            ],
            axis=1,
        ).dropna()

        if spec["higher_is_better"]:
            improvement = (paired["a"] - paired["b"]).to_numpy(dtype=float)
            definition = "model_a - model_b"
        else:
            improvement = (paired["b"] - paired["a"]).to_numpy(dtype=float)
            definition = "model_b - model_a (lower loss is better)"

        wins = int(np.sum(improvement > TIE_TOLERANCE))
        ties = int(np.sum(np.abs(improvement) <= TIE_TOLERANCE))
        losses = int(np.sum(improvement < -TIE_TOLERANCE))
        ci_low, ci_high = bootstrap_mean_ci(
            improvement,
            seed=BOOTSTRAP_SEED + pair_index,
        )
        statistic_two, p_two = safe_wilcoxon(improvement, alternative="two-sided")
        statistic_greater, p_greater = safe_wilcoxon(improvement, alternative="greater")

        pairwise_rows.append(
            {
                "model_a": model_a,
                "model_a_label": MODEL_LABEL[model_a],
                "model_b": model_b,
                "model_b_label": MODEL_LABEL[model_b],
                "metric": metric_name,
                "metric_label": spec["label"],
                "improvement_definition": definition,
                "n_paired": int(len(improvement)),
                "mean_improvement": float(np.mean(improvement)),
                "median_improvement": float(np.median(improvement)),
                "bootstrap_mean_ci_low": ci_low,
                "bootstrap_mean_ci_high": ci_high,
                "wins": wins,
                "ties": ties,
                "losses": losses,
                "win_rate_excluding_ties": wins / (wins + losses) if wins + losses else np.nan,
                "rank_biserial": rank_biserial_from_differences(improvement),
                "wilcoxon_stat_two_sided": statistic_two,
                "p_two_sided": p_two,
                "wilcoxon_stat_a_better": statistic_greater,
                "p_one_sided_a_better": p_greater,
            }
        )

pairwise_summary = pd.DataFrame(pairwise_rows)
pairwise_summary["p_two_sided_holm"] = holm_adjust(pairwise_summary["p_two_sided"])
pairwise_summary["p_one_sided_a_better_holm"] = holm_adjust(
    pairwise_summary["p_one_sided_a_better"]
)
pairwise_summary.to_csv(OUTPUT_DIR / "pairwise_comparisons.csv", index=False)

print_columns = [
    "model_a_label",
    "model_b_label",
    "metric_label",
    "n_paired",
    "mean_improvement",
    "bootstrap_mean_ci_low",
    "bootstrap_mean_ci_high",
    "wins",
    "ties",
    "losses",
    "rank_biserial",
    "p_two_sided_holm",
    "p_one_sided_a_better_holm",
]
print(pairwise_summary[print_columns].round(6).to_string(index=False))

In [ ]:
# Cell 6 - Friedman omnibus tests and average ranks
friedman_rows = []
rank_rows = []

for metric_name, spec in METRIC_SPECS.items():
    metric_table = pd.concat(
        [indexed[name][metric_name].rename(name) for name in MODEL_ORDER],
        axis=1,
    ).dropna()

    statistic, p_value = stats.friedmanchisquare(
        *[metric_table[name].to_numpy() for name in MODEL_ORDER]
    )
    per_dataset_ranks = metric_table.rank(
        axis=1,
        ascending=not spec["higher_is_better"],
        method="average",
    )

    friedman_rows.append(
        {
            "metric": metric_name,
            "metric_label": spec["label"],
            "n_complete_datasets": int(len(metric_table)),
            "friedman_statistic": float(statistic),
            "friedman_p_value": float(p_value),
        }
    )
    for model_name in MODEL_ORDER:
        rank_rows.append(
            {
                "metric": metric_name,
                "metric_label": spec["label"],
                "model": model_name,
                "model_label": MODEL_LABEL[model_name],
                "average_rank": float(per_dataset_ranks[model_name].mean()),
            }
        )

friedman_summary = pd.DataFrame(friedman_rows)
average_ranks = pd.DataFrame(rank_rows)
friedman_summary.to_csv(OUTPUT_DIR / "friedman_tests.csv", index=False)
average_ranks.to_csv(OUTPUT_DIR / "average_ranks.csv", index=False)

print("Friedman tests:")
print(friedman_summary.round(8).to_string(index=False))
print("\nAverage ranks (1 = best):")
print(average_ranks.round(6).to_string(index=False))

In [ ]:
# Cell 7 - build the auditable per-dataset comparison table and manifest
dataset_level = pd.DataFrame({"dataset": common_datasets})
dataset_level["n_samples"] = indexed[MODEL_ORDER[0]]["n_samples"].to_numpy()

for model_name in MODEL_ORDER:
    for metric_name in METRIC_SPECS:
        short_metric = metric_name.replace("test_", "")
        dataset_level[f"{model_name}_{short_metric}"] = indexed[model_name][
            metric_name
        ].to_numpy()

for model_a, model_b in PAIR_ORDER:
    for metric_name, spec in METRIC_SPECS.items():
        short_metric = metric_name.replace("test_", "")
        values_a = dataset_level[f"{model_a}_{short_metric}"]
        values_b = dataset_level[f"{model_b}_{short_metric}"]
        if spec["higher_is_better"]:
            improvement = values_a - values_b
        else:
            improvement = values_b - values_a
        dataset_level[
            f"improvement_{model_a}_vs_{model_b}_{short_metric}"
        ] = improvement

dataset_level.to_csv(OUTPUT_DIR / "dataset_level_comparison.csv", index=False)

manifest = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "seed": 42,
    "models": MODEL_ORDER,
    "model_labels": MODEL_LABEL,
    "source_files": {name: str(path) for name, path in RESULT_FILES.items()},
    "common_dataset_count": len(common_datasets),
    "expected_common_dataset_count": EXPECTED_COMMON_DATASETS,
    "excluded_consistently": ["Fungi"] if "Fungi" not in common_datasets else [],
    "tie_tolerance": TIE_TOLERANCE,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "improvement_direction": {
        "AUC": "model_a - model_b",
        "ACC": "model_a - model_b",
        "Loss": "model_b - model_a; positive favors model_a",
    },
}
with open(OUTPUT_DIR / "comparison_manifest.json", "w", encoding="utf-8") as handle:
    json.dump(manifest, handle, ensure_ascii=False, indent=2)

print("Dataset-level table rows:", len(dataset_level))
print(dataset_level.head(5).round(6).to_string(index=False))

In [ ]:
# Cell 8 - figures
COLORS = {
    "twotower": "#2563EB",
    "correlation": "#F59E0B",
    "transformer_only": "#64748B",
}


# Figure A: aggregate simple means
figure, axes = plt.subplots(1, 3, figsize=(13.0, 4.0))
aggregate_metric_map = [
    ("simple_mean_auc", "Mean AUC", True),
    ("simple_mean_acc", "Mean ACC", True),
    ("simple_mean_loss", "Mean BCE Loss", False),
]
for axis, (column, title, higher_better) in zip(axes, aggregate_metric_map):
    values = aggregate_summary.set_index("model").loc[MODEL_ORDER, column]
    bars = axis.bar(
        range(len(MODEL_ORDER)),
        values,
        color=[COLORS[name] for name in MODEL_ORDER],
    )
    axis.set_xticks(range(len(MODEL_ORDER)))
    axis.set_xticklabels([MODEL_LABEL[name] for name in MODEL_ORDER], rotation=18, ha="right")
    axis.set_title(title + (" ↑" if higher_better else " ↓"))
    axis.grid(axis="y", alpha=0.25)
    axis.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
figure.suptitle("V2 Ablation Aggregate Performance (124 Common Datasets)", y=1.03)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "aggregate_comparison.png", dpi=220, bbox_inches="tight")
plt.show()


# Figure B: win/tie/loss for Full Two-Tower against both ablations
full_pairs = pairwise_summary[pairwise_summary["model_a"] == "twotower"].copy()
full_pairs["comparison"] = (
    full_pairs["model_b_label"] + " — " + full_pairs["metric_label"]
)
figure, axis = plt.subplots(figsize=(9.0, 5.2))
y = np.arange(len(full_pairs))
axis.barh(y, full_pairs["wins"], label="Wins", color="#16A34A")
axis.barh(y, full_pairs["ties"], left=full_pairs["wins"], label="Ties", color="#CBD5E1")
axis.barh(
    y,
    full_pairs["losses"],
    left=full_pairs["wins"] + full_pairs["ties"],
    label="Losses",
    color="#DC2626",
)
axis.set_yticks(y)
axis.set_yticklabels(full_pairs["comparison"])
axis.set_xlabel("Number of paired datasets")
axis.set_title("Fractional Two-Tower: Win / Tie / Loss")
axis.legend(ncol=3, loc="lower right")
axis.grid(axis="x", alpha=0.2)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "full_model_win_tie_loss.png", dpi=220, bbox_inches="tight")
plt.show()


# Figure C: sorted per-dataset improvements; positive favors Fractional Two-Tower
baselines = ["correlation", "transformer_only"]
metric_keys = ["auc", "acc", "loss"]
figure, axes = plt.subplots(2, 3, figsize=(15.0, 7.2), sharex=False)
for row_index, baseline in enumerate(baselines):
    for col_index, metric_key in enumerate(metric_keys):
        axis = axes[row_index, col_index]
        column = f"improvement_twotower_vs_{baseline}_{metric_key}"
        values = dataset_level[column].dropna().sort_values().to_numpy()
        colors = np.where(values >= 0, "#2563EB", "#DC2626")
        axis.bar(np.arange(len(values)), values, color=colors, width=1.0)
        axis.axhline(0.0, color="black", linewidth=0.8)
        axis.set_title(f"vs {MODEL_LABEL[baseline]} — {metric_key.upper()}")
        axis.set_ylabel("Improvement (positive = Full better)")
        axis.set_xlabel("Datasets sorted by improvement")
        axis.grid(axis="y", alpha=0.2)
figure.suptitle("Fractional Two-Tower Per-Dataset Improvements", y=1.02)
figure.tight_layout()
figure.savefig(FIGURE_DIR / "full_model_sorted_improvements.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9 - final file inventory and concise conclusion checks
generated_files = sorted(
    [path for path in OUTPUT_DIR.rglob("*") if path.is_file()],
    key=lambda path: str(path),
)
print("Generated files:")
for path in generated_files:
    print(" -", path)

full_vs = pairwise_summary[pairwise_summary["model_a"] == "twotower"].copy()
print("\nFractional Two-Tower directional checks:")
for _, row in full_vs.iterrows():
    ci_excludes_zero = row["bootstrap_mean_ci_low"] > 0 or row["bootstrap_mean_ci_high"] < 0
    print(
        f"vs {row['model_b_label']} | {row['metric_label']} | "
        f"mean improvement={row['mean_improvement']:.6f} | "
        f"W/T/L={int(row['wins'])}/{int(row['ties'])}/{int(row['losses'])} | "
        f"Holm one-sided p={row['p_one_sided_a_better_holm']:.6g} | "
        f"bootstrap CI excludes 0={ci_excludes_zero}"
    )

print(
    "\nInterpret results from pairwise_comparisons.csv. Aggregate superiority alone "
    "does not imply that the full model wins on every dataset."
)